# Hyperspectral Super-Resolution — Training Pipeline

**RRDBNet6x** (modified ESRGAN) for 6× EMIT→Sentinel-2 fusion GT upsampling.

Notebook structure:
1. Setup & Drive
2. **Configuration** ← edit this cell only
3. Data pipeline (flexible GT source + .npy cache)
4. Model & dataset registration
5. Training
6. Evaluation & visualization

## 1. Setup

In [ ]:
# ── Install BasicSR + dependencies ──
!git clone https://github.com/XPixelGroup/BasicSR.git 2>/dev/null || echo 'BasicSR already cloned'
%cd /content/BasicSR
!pip install -e . -q
!pip install rasterio joblib scikit-learn wandb -q

import sys
sys.path.insert(0, '/content/BasicSR')

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, textwrap
os.environ['GH_USER'] = userdata.get('GH_USER')
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['EARTHDATA_USERNAME'] = userdata.get('EARTHDATA_USERNAME')
os.environ['EARTHDATA_PASSWORD'] = userdata.get('EARTHDATA_PASSWORD')

# Git credential helper for private repo clone
askpass_path = '/tmp/gh_askpass.sh'
with open(askpass_path, 'w') as f:
    f.write(textwrap.dedent('''\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    '''))
os.chmod(askpass_path, 0o700)
os.environ['GIT_ASKPASS'] = askpass_path
os.environ['GIT_TERMINAL_PROMPT'] = '0'

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git 2>/dev/null || echo 'Repo already cloned'

## 2. Configuration — edit this cell only

In [ ]:
from pathlib import Path
from datetime import datetime


class CFG:
    # ── Data source ──────────────────────────────────────────
    DRIVE_ROOT  = Path('/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches')
    DATA_DATE   = '2026-03-22'           # date folder under DRIVE_ROOT

    GT_SOURCE   = 'regression'           # 'regression', 'sfim', 'glp', or any method
    R2_THRESH   = 0.75                   # R² filter (only when GT_SOURCE='regression')
    MAX_AOIS    = None                   # limit AOI count (None = use all available)

    # ── .npy cache ───────────────────────────────────────────
    USE_CACHE   = True                   # try loading pre-built .npy zip from Drive
    SAVE_CACHE  = True                   # save .npy zip after building from GeoTIFF
    CACHE_ZIP   = None                   # explicit zip path override (None = auto-name)

    # ── Model ────────────────────────────────────────────────
    EXP_NAME    = None                   # None = auto-generate from GT_SOURCE + arch
    NUM_BANDS   = 32
    SCALE       = 6
    GT_SIZE     = 300                    # must be divisible by SCALE
    BATCH_SIZE  = 4
    TOTAL_ITER  = 100_000
    LR_RATE     = 2e-4
    NUM_FEAT    = 128
    NUM_BLOCK   = 16
    MILESTONES  = [30_000, 60_000, 90_000]

    # ── Resume ───────────────────────────────────────────────
    RESUME_STATE = None                  # path to .state file, or None
    PRETRAIN_G   = None                  # path to pretrained net_g .pth, or None

    # ── Logging ──────────────────────────────────────────────
    VAL_FREQ        = 1000
    SAVE_CKPT_FREQ  = 10_000
    N_VIS_SAMPLES   = 4                  # fixed tiles for WandB reconstruction viz
    VIS_BANDS       = (10, 6, 2)         # bands for RGB composite
    WANDB_PROJECT   = 'EMIT-S2-SuperResolution'
    WANDB_ENTITY    = None

    # ── Constants (do not edit) ──────────────────────────────
    LOCAL_DATA  = Path('/content/data_local')
    EMIT_SCALE  = 10_000.0
    EMIT_NODATA = 65535


# ── Derived values ────────────────────────────────────────────
if CFG.EXP_NAME is None:
    _aoi_tag = f'-{CFG.MAX_AOIS}aoi' if CFG.MAX_AOIS else ''
    CFG.EXP_NAME = f'{CFG.GT_SOURCE}-{CFG.NUM_FEAT}x{CFG.NUM_BLOCK}{_aoi_tag}-{datetime.now():%m%d}'

CFG.DATA_DIR = CFG.DRIVE_ROOT / CFG.DATA_DATE
CFG.EXP_DIR  = CFG.DRIVE_ROOT / CFG.EXP_NAME

if CFG.CACHE_ZIP is None:
    tag = f'{CFG.GT_SOURCE}_r2-{CFG.R2_THRESH}' if CFG.GT_SOURCE == 'regression' else CFG.GT_SOURCE
    if CFG.MAX_AOIS:
        tag += f'_{CFG.MAX_AOIS}aoi'
    CFG.CACHE_ZIP = CFG.DRIVE_ROOT / 'caches' / f'{tag}_npy.zip'
else:
    CFG.CACHE_ZIP = Path(CFG.CACHE_ZIP)

# R² CSV exported by Color_Matching.ipynb
CFG.R2_CSV = CFG.DATA_DIR / 'r2_all_tiles.csv'

# ── Validate ───────────────────────────────────────────────────
assert CFG.GT_SIZE % CFG.SCALE == 0, \
    f'GT_SIZE={CFG.GT_SIZE} must be divisible by SCALE={CFG.SCALE}'

print(f'Experiment : {CFG.EXP_NAME}')
print(f'GT source  : {CFG.GT_SOURCE}' + (f' (R²≥{CFG.R2_THRESH})' if CFG.GT_SOURCE == 'regression' else ''))
print(f'Max AOIs   : {CFG.MAX_AOIS or "all"}')
print(f'LR crop    : {CFG.GT_SIZE // CFG.SCALE}×{CFG.GT_SIZE // CFG.SCALE}')
print(f'GT crop    : {CFG.GT_SIZE}×{CFG.GT_SIZE}')
print(f'Network    : RRDBNet6x  {CFG.NUM_FEAT}f / {CFG.NUM_BLOCK}b')
print(f'R² CSV     : {CFG.R2_CSV}' + (' ✓' if CFG.R2_CSV.exists() else ' (not found — will compute)'))
print(f'Cache      : {CFG.CACHE_ZIP}')
print(f'Output     : {CFG.EXP_DIR}')

In [ ]:
import wandb
wandb.login()
print(f'WandB ready — project: {CFG.WANDB_PROJECT}')

## 3. Data pipeline

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import re, shutil
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# ── GT source registry ──────────────────────────────────────
# Maps GT_SOURCE name → (subdirectory under pair_dir, glob pattern)
# Add new fusion methods here.
GT_SOURCES = {
    'regression': ('regression_synth',  '*_regression_synth.tif'),
    'sfim':       ('SFIM_Bilinear',     '*_SFIM_fused.tif'),
    'glp':        ('GLP',               '*_GLP_fused.tif'),
}


# ── Scene discovery ─────────────────────────────────────────
def discover_scenes(data_dir):
    """Scan two-level deep: aoi_*/pair_id/ → dict of scene paths."""
    scenes = {}
    for aoi_dir in sorted(data_dir.glob('aoi_*')):
        for pair_dir in sorted(aoi_dir.iterdir()):
            if not pair_dir.is_dir():
                continue
            key = f'{aoi_dir.name}__{pair_dir.name}'
            scenes[key] = {'tiles': pair_dir / 'tiles', 'aoi': aoi_dir.name, 'pair': pair_dir}
    return scenes


# ── R² loading ──────────────────────────────────────────────
def load_r2_from_csv(csv_path):
    """Load R² values from Color_Matching.ipynb's r2_all_tiles.csv.

    Returns dict: (scene_key, tile_id) → r2_mean
    where scene_key = 'aoi_slug__pair_id' to match discover_scenes() keys.
    """
    df = pd.read_csv(csv_path)
    r2_map = {}
    for _, row in df.iterrows():
        scene_key = f"{row['aoi_slug']}__{row['pair_id']}"
        r2_map[(scene_key, int(row['tile_idx']))] = float(row['r2_mean'])
    kept = sum(1 for v in r2_map.values() if v >= CFG.R2_THRESH)
    print(f'R² loaded from CSV: {len(r2_map)} tiles, {kept} with R²≥{CFG.R2_THRESH}')
    return r2_map


def compute_r2_from_tif(scenes, gt_subdir, gt_pattern):
    """Fallback: compute R² from GeoTIFFs when CSV is not available."""
    tasks = []
    for scene_key, paths in tqdm(scenes.items(), desc='Scanning'):
        tile_dir = paths['tiles']
        gt_dir = paths['pair'] / gt_subdir
        if not gt_dir.exists():
            continue

        emit_by_id = {int(m.group(1)): p for p in tile_dir.glob('*_emit_b32.tif')
                      if (m := re.search(r'tile(\d+)', p.stem))}
        synth_by_id = {int(m.group(1)): p for p in gt_dir.glob(gt_pattern)
                       if (m := re.search(r'tile(\d+)', p.stem))}

        for tid in emit_by_id:
            if tid in synth_by_id:
                tasks.append((scene_key, tid, synth_by_id[tid], emit_by_id[tid]))

    print(f'Computing R² for {len(tasks)} tile pairs (CSV not found)...')
    r2_map = {}

    def _compute_tile_r2(synth_path, emit_b32_path, scale=6):
        with rasterio.open(synth_path) as ds:
            synth = ds.read().astype(np.float32)
        with rasterio.open(emit_b32_path) as ds:
            emit = ds.read().astype(np.float32)
        emit_up = np.repeat(np.repeat(emit, scale, axis=1), scale, axis=2)
        valid = ((synth != CFG.EMIT_NODATA) & (emit_up != CFG.EMIT_NODATA)
                 & (emit_up != 0)).all(axis=0)
        if valid.sum() < 100:
            return -1.0
        r2s = []
        for b in range(synth.shape[0]):
            yt, yp = emit_up[b][valid], synth[b][valid]
            ss_res, ss_tot = np.sum((yt - yp)**2), np.sum((yt - yt.mean())**2)
            if ss_tot > 0: r2s.append(1.0 - ss_res / ss_tot)
        return float(np.mean(r2s)) if r2s else -1.0

    def _compute(args):
        sk, tid, sp, ep = args
        try: return (sk, tid, _compute_tile_r2(sp, ep, scale=CFG.SCALE))
        except Exception: return (sk, tid, -1.0)

    with ThreadPoolExecutor(max_workers=8) as pool:
        futs = [pool.submit(_compute, t) for t in tasks]
        for f in tqdm(as_completed(futs), total=len(tasks), desc='R²'):
            sk, tid, r2 = f.result()
            r2_map[(sk, tid)] = r2

    kept = sum(1 for v in r2_map.values() if v >= CFG.R2_THRESH)
    print(f'R² computed: {len(r2_map)} tiles, {kept} with R²≥{CFG.R2_THRESH}')
    return r2_map


# ── Read helpers ────────────────────────────────────────────
def read_emit_b32(path):
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    mask = arr == CFG.EMIT_NODATA
    arr /= CFG.EMIT_SCALE
    arr[mask] = 0.0
    return arr


def read_gt(path):
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    arr /= CFG.EMIT_SCALE
    return np.clip(np.nan_to_num(arr, nan=0.0), 0.0, 1.5)


def save_pair(lr, gt, stem, split):
    assert lr.shape[0] == gt.shape[0], f'Band mismatch: {lr.shape} vs {gt.shape}'
    lr = np.nan_to_num(lr, nan=0.0, posinf=0.0, neginf=0.0)
    gt = np.clip(np.nan_to_num(gt, nan=0.0, posinf=0.0, neginf=0.0), 0.0, 1.5)
    np.save(CFG.LOCAL_DATA / split / 'lq' / f'{stem}.npy', lr)
    np.save(CFG.LOCAL_DATA / split / 'gt' / f'{stem}.npy', gt)


# ── AOI split (with optional MAX_AOIS limit) ───────────────
def split_aois(scenes, seed=42):
    all_aois = sorted(set(v['aoi'] for v in scenes.values()))
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(all_aois))

    # Apply MAX_AOIS limit before splitting
    if CFG.MAX_AOIS and CFG.MAX_AOIS < len(all_aois):
        perm = perm[:CFG.MAX_AOIS]
        print(f'MAX_AOIS={CFG.MAX_AOIS}: using {len(perm)} of {len(all_aois)} AOIs')

    selected = [all_aois[i] for i in perm]
    n = len(selected)
    n_test = max(1, int(0.15 * n))
    n_val  = max(1, int(0.15 * n))

    test_aois  = set(selected[:n_test])
    val_aois   = set(selected[n_test:n_test + n_val])
    train_aois = set(selected[n_test + n_val:])

    print(f'AOI split: {n} selected → {len(train_aois)} train / {len(val_aois)} val / {len(test_aois)} test')
    return train_aois, val_aois, test_aois


# ── Main data preparation ──────────────────────────────────
def prepare_data_from_tif():
    """Build .npy pairs from GeoTIFF tiles on Drive."""
    gt_sub, gt_pat = GT_SOURCES[CFG.GT_SOURCE]
    scenes = discover_scenes(CFG.DATA_DIR)
    train_aois, val_aois, test_aois = split_aois(scenes)
    used_aois = train_aois | val_aois | test_aois

    # R² filter (only for regression GT)
    r2_map = {}
    if CFG.GT_SOURCE == 'regression':
        if CFG.R2_CSV.exists():
            r2_map = load_r2_from_csv(CFG.R2_CSV)
        else:
            r2_map = compute_r2_from_tif(scenes, gt_sub, gt_pat)

    for split in ['train', 'val', 'test']:
        (CFG.LOCAL_DATA / split / 'lq').mkdir(parents=True, exist_ok=True)
        (CFG.LOCAL_DATA / split / 'gt').mkdir(parents=True, exist_ok=True)

    counts = {'train': 0, 'val': 0, 'test': 0}
    skipped = {'r2': 0, 'missing': 0, 'error': 0, 'aoi': 0}

    for scene_key, paths in tqdm(scenes.items(), desc='Building .npy'):
        aoi = paths['aoi']
        if aoi not in used_aois:
            skipped['aoi'] += 1
            continue
        split = 'test' if aoi in test_aois else 'val' if aoi in val_aois else 'train'

        gt_dir = paths['pair'] / gt_sub
        if not gt_dir.exists():
            continue

        gt_by_id = {int(m.group(1)): p for p in gt_dir.glob(gt_pat)
                    if (m := re.search(r'tile(\d+)', p.stem))}

        for lr_path in sorted(paths['tiles'].glob('*_emit_b32.tif')):
            m = re.search(r'tile(\d+)', lr_path.stem)
            if not m:
                continue
            tid = int(m.group(1))

            if tid not in gt_by_id:
                skipped['missing'] += 1
                continue

            if CFG.GT_SOURCE == 'regression':
                if r2_map.get((scene_key, tid), -1.0) < CFG.R2_THRESH:
                    skipped['r2'] += 1
                    continue

            try:
                lr = read_emit_b32(lr_path)
                gt = read_gt(gt_by_id[tid])
                save_pair(lr, gt, f'{scene_key}__tile{tid:03d}', split)
                counts[split] += 1
            except Exception as e:
                skipped['error'] += 1
                print(f'  ERROR {scene_key}/tile{tid}: {e}')

    print(f'\nDone: {sum(counts.values())} tiles')
    for s, c in counts.items():
        print(f'  {s}: {c}')
    if skipped['aoi']:     print(f'  Skipped (AOI limit): {skipped["aoi"]} scenes')
    if skipped['r2']:      print(f'  Skipped (R²<{CFG.R2_THRESH}): {skipped["r2"]}')
    if skipped['missing']: print(f'  Skipped (no GT): {skipped["missing"]}')
    if skipped['error']:   print(f'  Errors: {skipped["error"]}')


# ── Cache management ─────────────────────────────────────────
def load_cache():
    """Unzip cached .npy data from Drive → local."""
    if not CFG.CACHE_ZIP.exists():
        print(f'No cache at {CFG.CACHE_ZIP}')
        return False
    print(f'Loading cache: {CFG.CACHE_ZIP}')
    if CFG.LOCAL_DATA.exists():
        shutil.rmtree(CFG.LOCAL_DATA)
    shutil.unpack_archive(CFG.CACHE_ZIP, CFG.LOCAL_DATA)
    for split in ['train', 'val', 'test']:
        n = len(list((CFG.LOCAL_DATA / split / 'gt').glob('*.npy')))
        print(f'  {split}: {n} tiles')
    return True


def save_cache():
    """Zip local .npy data → Drive for reuse."""
    CFG.CACHE_ZIP.parent.mkdir(parents=True, exist_ok=True)
    tmp = str(CFG.CACHE_ZIP).replace('.zip', '')
    shutil.make_archive(tmp, 'zip', CFG.LOCAL_DATA)
    print(f'Cache saved: {CFG.CACHE_ZIP}  ({CFG.CACHE_ZIP.stat().st_size / 1e9:.2f} GB)')


# ── Execute ──────────────────────────────────────────────────
loaded = False
if CFG.USE_CACHE:
    loaded = load_cache()

if not loaded:
    assert CFG.GT_SOURCE in GT_SOURCES, \
        f'Unknown GT_SOURCE={CFG.GT_SOURCE!r}. Known: {list(GT_SOURCES.keys())}'
    prepare_data_from_tif()
    if CFG.SAVE_CACHE:
        save_cache()

## 4. Model & dataset registration

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from basicsr.utils.registry import DATASET_REGISTRY
import numpy as np, random
from pathlib import Path


if 'PairedNpyDataset' in DATASET_REGISTRY._obj_map:
    del DATASET_REGISTRY._obj_map['PairedNpyDataset']


@DATASET_REGISTRY.register()
class PairedNpyDataset(Dataset):
    """Paired LR/GT .npy dataset with LR-aligned random cropping."""

    def __init__(self, opt):
        super().__init__()
        self.opt = opt
        self.gt_dir = Path(opt['dataroot_gt'])
        self.lq_dir = Path(opt['dataroot_lq'])
        self.scale = opt.get('scale', 6)
        self.gt_size = opt.get('gt_size', 192)

        self.gt_files = sorted(self.gt_dir.glob('*.npy'))
        self.lq_files = sorted(self.lq_dir.glob('*.npy'))
        assert len(self.gt_files) == len(self.lq_files) > 0, \
            f'Mismatch or empty: {len(self.gt_files)} GT vs {len(self.lq_files)} LQ in {self.gt_dir}'
        for g, l in zip(self.gt_files, self.lq_files):
            assert g.stem == l.stem, f'Name mismatch: {g.stem} vs {l.stem}'

        print(f'[PairedNpyDataset] {len(self.gt_files)} pairs, scale={self.scale}, gt_size={self.gt_size}')

    def __len__(self):
        return len(self.gt_files)

    def __getitem__(self, idx):
        gt = np.load(self.gt_files[idx]).astype(np.float32)
        lq = np.load(self.lq_files[idx]).astype(np.float32)

        if self.opt.get('phase') == 'train' and self.gt_size:
            _, H_gt, W_gt = gt.shape
            gt_size = self.gt_size
            lq_size = gt_size // self.scale

            # Crop in LR space first, then scale to GT (avoids sub-pixel misalignment)
            top_lq  = random.randint(0, max(0, H_gt // self.scale - lq_size))
            left_lq = random.randint(0, max(0, W_gt // self.scale - lq_size))
            top_gt, left_gt = top_lq * self.scale, left_lq * self.scale

            gt = gt[:, top_gt:top_gt + gt_size, left_gt:left_gt + gt_size]
            lq = lq[:, top_lq:top_lq + lq_size, left_lq:left_lq + lq_size]

            # Augmentation: horizontal flip + vertical flip + 90° rotation
            if self.opt.get('use_hflip', False) and random.random() > 0.5:
                gt = np.flip(gt, axis=2).copy()
                lq = np.flip(lq, axis=2).copy()
            if self.opt.get('use_rot', False) and random.random() > 0.5:
                gt = np.flip(gt, axis=1).copy()
                lq = np.flip(lq, axis=1).copy()
            if self.opt.get('use_rot', False) and random.random() > 0.5:
                gt = np.rot90(gt, k=1, axes=(1, 2)).copy()
                lq = np.rot90(lq, k=1, axes=(1, 2)).copy()

        return {
            'lq': torch.from_numpy(lq.copy()).float(),
            'gt': torch.from_numpy(gt.copy()).float(),
            'lq_path': str(self.lq_files[idx]),
            'gt_path': str(self.gt_files[idx]),
        }


print('PairedNpyDataset registered.')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from basicsr.archs.rrdbnet_arch import RRDB
from basicsr.utils.registry import ARCH_REGISTRY


if 'RRDBNet6x' in ARCH_REGISTRY._obj_map:
    del ARCH_REGISTRY._obj_map['RRDBNet6x']


@ARCH_REGISTRY.register()
class RRDBNet6x(nn.Module):
    """RRDBNet for 6× upsampling: 2× nearest + 3× nearest = 6× total.
    Bicubic skip connection: output = learned_residual + bicubic(input).
    conv_last is zero-initialized so initial output = pure bicubic.
    """

    def __init__(self, num_in_ch, num_out_ch, num_feat=64, num_block=23, num_grow_ch=32):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)

        self.body = nn.ModuleList([RRDB(num_feat, num_grow_ch=num_grow_ch)
                                   for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        self.conv_up1  = nn.Conv2d(num_feat, num_feat, 3, 1, 1)  # after 2× upsample
        self.conv_up2  = nn.Conv2d(num_feat, num_feat, 3, 1, 1)  # after 3× upsample
        self.conv_hr   = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)

        # Zero-init last conv → initial output is pure bicubic skip
        nn.init.zeros_(self.conv_last.weight)
        nn.init.zeros_(self.conv_last.bias)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = feat
        for block in self.body:
            body_feat = block(body_feat)
        feat = feat + self.conv_body(body_feat)

        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode='nearest')))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=3, mode='nearest')))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))

        base = F.interpolate(x, scale_factor=6, mode='bicubic', align_corners=False)
        return out + base  # pure additive skip — do NOT scale


# Quick shape test
_net = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=64, num_block=4)
_x = torch.randn(1, 32, 20, 20)
_y = _net(_x)
assert _y.shape == (1, 32, 120, 120), f'Expected (1,32,120,120), got {_y.shape}'
del _net, _x, _y
print(f'RRDBNet6x registered.  20×20 → 120×120 ✓')

In [ ]:
import importlib
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for validation logging
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
import wandb

from basicsr.models.sr_model import SRModel
from basicsr.utils.registry import MODEL_REGISTRY
import basicsr.utils.img_util as _iu
import basicsr.models.sr_model as _srm


# ── Patch tensor2img for >4 channel data ─────────────────────
# BasicSR assumes 3-ch RGB and does rgb→bgr conversion which
# crashes on 32-band hyperspectral data.
importlib.reload(_iu)
_orig_t2i = _iu.tensor2img

def _patched_tensor2img(tensor, rgb2bgr=True, out_type=np.uint8, min_max=(0, 1)):
    t = tensor[0] if isinstance(tensor, list) else tensor
    if t.ndim >= 3 and t.shape[-3] > 4:
        rgb2bgr = False
    return _orig_t2i(tensor, rgb2bgr=rgb2bgr, out_type=out_type, min_max=min_max)

_iu.tensor2img = _patched_tensor2img
_srm.tensor2img = _patched_tensor2img


# ── Visualization helpers ─────────────────────────────────────
def to_rgb(cube, bands=CFG.VIS_BANDS):
    """Convert (C,H,W) hyperspectral cube → (H,W,3) RGB for display."""
    rgb = np.stack([cube[b] for b in bands], axis=-1)
    pos = rgb[rgb > 0]
    if len(pos) > 0:
        lo, hi = np.percentile(pos, [2, 98])
    else:
        lo, hi = 0.0, 1.0
    return np.clip((rgb - lo) / (hi - lo + 1e-8), 0, 1)


def make_vis_figure(lq_np, sr_np, gt_np, scale, bands):
    """Create a 2×3 figure: [Bicubic | SR | GT] / [RMSE | SAM | Per-band error]."""
    bic_np = F.interpolate(
        torch.from_numpy(lq_np[None]), scale_factor=scale,
        mode='bicubic', align_corners=False
    ).squeeze(0).numpy()

    sr_rgb  = to_rgb(sr_np,  bands)
    gt_rgb  = to_rgb(gt_np,  bands)
    bic_rgb = to_rgb(bic_np, bands)

    # Error maps
    rmse_map = np.sqrt(np.mean((sr_np - gt_np) ** 2, axis=0))
    dot   = np.sum(sr_np * gt_np, axis=0)
    n_sr  = np.linalg.norm(sr_np, axis=0).clip(1e-8)
    n_gt  = np.linalg.norm(gt_np, axis=0).clip(1e-8)
    sam_map = np.degrees(np.arccos(np.clip(dot / (n_sr * n_gt), -1 + 1e-7, 1 - 1e-7)))

    # Per-band RMSE
    band_rmse_sr  = np.sqrt(np.mean((sr_np  - gt_np) ** 2, axis=(1, 2)))
    band_rmse_bic = np.sqrt(np.mean((bic_np - gt_np) ** 2, axis=(1, 2)))

    # Metrics
    cb = scale
    sc = sr_np[:, cb:-cb, cb:-cb]
    gc = gt_np[:, cb:-cb, cb:-cb]
    bc = bic_np[:, cb:-cb, cb:-cb]
    mse_sr  = np.mean((sc - gc) ** 2)
    mse_bic = np.mean((bc - gc) ** 2)
    psnr_sr  = 10 * np.log10(1.0 / mse_sr)  if mse_sr  > 0 else 100.0
    psnr_bic = 10 * np.log10(1.0 / mse_bic) if mse_bic > 0 else 100.0

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes[0, 0].imshow(bic_rgb)
    axes[0, 0].set_title(f'Bicubic ×{scale}\nPSNR={psnr_bic:.2f} dB', fontsize=9)
    axes[0, 1].imshow(sr_rgb)
    axes[0, 1].set_title(f'SR\nPSNR={psnr_sr:.2f} dB', fontsize=9)
    axes[0, 2].imshow(gt_rgb)
    axes[0, 2].set_title('GT', fontsize=9)

    im1 = axes[1, 0].imshow(rmse_map, cmap='hot', vmin=0)
    axes[1, 0].set_title('RMSE (per pixel)', fontsize=9)
    plt.colorbar(im1, ax=axes[1, 0], fraction=0.046)

    im2 = axes[1, 1].imshow(sam_map, cmap='hot', vmin=0)
    axes[1, 1].set_title('SAM (degrees)', fontsize=9)
    plt.colorbar(im2, ax=axes[1, 1], fraction=0.046)

    axes[1, 2].plot(band_rmse_bic, 'o-', ms=3, color='#FF9800', label='Bicubic', alpha=0.7)
    axes[1, 2].plot(band_rmse_sr,  's-', ms=3, color='#2196F3', label='SR', alpha=0.7)
    axes[1, 2].set_xlabel('Band')
    axes[1, 2].set_ylabel('RMSE')
    axes[1, 2].set_title('Per-band RMSE', fontsize=9)
    axes[1, 2].legend(fontsize=8)
    axes[1, 2].grid(True, alpha=0.3)

    for ax in axes[:2, :3].flat:
        if ax != axes[1, 2]:
            ax.axis('off')
    plt.tight_layout()
    return fig


# ── Custom SRModel with WandB reconstruction logging ─────────
if 'WandBSRModel' in MODEL_REGISTRY._obj_map:
    del MODEL_REGISTRY._obj_map['WandBSRModel']


@MODEL_REGISTRY.register()
class WandBSRModel(SRModel):
    """SRModel that logs reconstruction visualizations to WandB
    every validation step (RMSE heatmap + SAM map + per-band error)."""

    def nondist_validation(self, dataloader, current_iter, tb_logger, save_img):
        # Run standard BasicSR validation (metrics + optional image save)
        super().nondist_validation(dataloader, current_iter, tb_logger, save_img)

        # Skip WandB logging if wandb not active
        if wandb.run is None:
            return

        # Pick N fixed tiles (evenly spaced, deterministic)
        dataset = dataloader.dataset
        n = len(dataset)
        n_vis = min(CFG.N_VIS_SAMPLES, n)
        indices = np.linspace(0, n - 1, n_vis, dtype=int).tolist()

        images = {}
        for i, idx in enumerate(indices):
            data = dataset[idx]
            lq = data['lq'].unsqueeze(0).to(self.device)

            with torch.no_grad():
                sr = self.net_g(lq)

            lq_np = data['lq'].numpy()
            sr_np = sr.squeeze(0).cpu().numpy()
            gt_np = data['gt'].numpy()

            fig = make_vis_figure(lq_np, sr_np, gt_np, CFG.SCALE, CFG.VIS_BANDS)
            images[f'val_vis/tile_{i}'] = wandb.Image(fig)
            plt.close(fig)

        wandb.log(images, step=current_iter)


print('WandBSRModel registered.')

## 5. Training

In [ ]:
import yaml
from pathlib import Path

config = {
    'name': CFG.EXP_NAME,
    'model_type': 'WandBSRModel',
    'scale': CFG.SCALE,
    'num_gpu': 1,
    'manual_seed': 42,

    'datasets': {
        'train': {
            'name': 'EMIT_S2_train',
            'type': 'PairedNpyDataset',
            'dataroot_gt': str(CFG.LOCAL_DATA / 'train' / 'gt'),
            'dataroot_lq': str(CFG.LOCAL_DATA / 'train' / 'lq'),
            'gt_size': CFG.GT_SIZE,
            'scale': CFG.SCALE,
            'use_hflip': True,
            'use_rot': True,
            'num_worker_per_gpu': 2,
            'batch_size_per_gpu': CFG.BATCH_SIZE,
            'dataset_enlarge_ratio': 100,
        },
        'val': {
            'name': 'EMIT_S2_val',
            'type': 'PairedNpyDataset',
            'dataroot_gt': str(CFG.LOCAL_DATA / 'val' / 'gt'),
            'dataroot_lq': str(CFG.LOCAL_DATA / 'val' / 'lq'),
            'scale': CFG.SCALE,
            'io_backend': {'type': 'disk'},
        },
    },

    'network_g': {
        'type': 'RRDBNet6x',
        'num_in_ch': CFG.NUM_BANDS,
        'num_out_ch': CFG.NUM_BANDS,
        'num_feat': CFG.NUM_FEAT,
        'num_block': CFG.NUM_BLOCK,
        'num_grow_ch': 32,
    },

    'path': {
        'pretrain_network_g': str(CFG.PRETRAIN_G) if CFG.PRETRAIN_G else None,
        'strict_load_g': True,
        'resume_state': str(CFG.RESUME_STATE) if CFG.RESUME_STATE else None,
        'experiments_root': str(CFG.EXP_DIR),
        'models': str(CFG.EXP_DIR / 'models'),
        'training_states': str(CFG.EXP_DIR / 'training_states'),
        'log': str(CFG.EXP_DIR),
        'visualization': str(CFG.EXP_DIR / 'visualization'),
    },

    'train': {
        'ema_decay': 0.999,
        'optim_g': {
            'type': 'Adam',
            'lr': CFG.LR_RATE,
            'weight_decay': 0,
            'betas': [0.9, 0.99],
        },
        'scheduler': {
            'type': 'MultiStepLR',
            'milestones': CFG.MILESTONES,
            'gamma': 0.5,
        },
        'total_iter': CFG.TOTAL_ITER,
        'warmup_iter': -1,
        'grad_clip': {'type': 'norm', 'max_norm': 5.0},
        'pixel_opt': {
            'type': 'L1Loss',
            'loss_weight': 1.0,
            'reduction': 'mean',
        },
    },

    'val': {
        'val_freq': CFG.VAL_FREQ,
        'save_img': False,
        'metrics': {
            'psnr': {
                'type': 'calculate_psnr',
                'crop_border': CFG.SCALE,
                'test_y_channel': False,
            },
        },
    },

    'logger': {
        'print_freq': 100,
        'save_checkpoint_freq': CFG.SAVE_CKPT_FREQ,
        'use_tb_logger': True,
        'wandb': {
            'project': CFG.WANDB_PROJECT,
            'resume_id': None,
        },
    },

    'dist_params': {'backend': 'nccl', 'port': 29500},
}

config_path = Path('/content/BasicSR/options/train') / f'{CFG.EXP_NAME}.yml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f'Config: {config_path}')
print(f'  GT crop:   {CFG.GT_SIZE}×{CFG.GT_SIZE}  →  LR crop: {CFG.GT_SIZE // CFG.SCALE}×{CFG.GT_SIZE // CFG.SCALE}')
print(f'  Network:   RRDBNet6x  {CFG.NUM_FEAT}f / {CFG.NUM_BLOCK}b')
print(f'  Iters:     {CFG.TOTAL_ITER:,}   batch: {CFG.BATCH_SIZE}')
print(f'  LR:        {CFG.LR_RATE}  milestones: {CFG.MILESTONES}')
print(f'  Val freq:  {CFG.VAL_FREQ}   Ckpt freq: {CFG.SAVE_CKPT_FREQ}')

In [ ]:
import sys
sys.path.insert(0, '/content/BasicSR')
sys.argv = ['train.py', '-opt', str(config_path)]

from basicsr.train import train_pipeline
train_pipeline('/content/BasicSR')

## 6. Post-training evaluation

In [ ]:
# ── Bicubic baseline (sanity check) ─────────────────────────
import torch, torch.nn.functional as F, numpy as np
from pathlib import Path

val_lq = sorted((CFG.LOCAL_DATA / 'val' / 'lq').glob('*.npy'))
val_gt = sorted((CFG.LOCAL_DATA / 'val' / 'gt').glob('*.npy'))

psnrs = []
for lq_p, gt_p in zip(val_lq, val_gt):
    lq = np.load(lq_p)
    gt = np.load(gt_p)
    bic = F.interpolate(torch.from_numpy(lq[None]).float(),
                        scale_factor=6, mode='bicubic',
                        align_corners=False).squeeze(0).numpy()
    cb = CFG.SCALE
    mse = np.mean((gt[:, cb:-cb, cb:-cb] - bic[:, cb:-cb, cb:-cb]) ** 2)
    if mse > 0:
        psnrs.append(10 * np.log10(1.0 / mse))

print(f'Bicubic baseline (val, crop_border={CFG.SCALE}): {np.mean(psnrs):.2f} ± {np.std(psnrs):.2f} dB')

In [ ]:
import re, csv, torch, numpy as np, matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm


def find_checkpoint(exp_dir, exp_name):
    """Find the latest numbered net_g_*.pth checkpoint."""
    search = [Path(exp_dir) / 'models',
              Path(exp_dir) / exp_name / 'models',
              Path('/content/BasicSR/experiments') / exp_name / 'models']
    ckpts = []
    for d in search:
        if d.exists():
            ckpts.extend(d.glob('net_g_*.pth'))
    numbered = [(p, int(m.group(1))) for p in ckpts
                if (m := re.search(r'(\d+)', p.stem))]
    if not numbered:
        print(f'No checkpoints found in: {[str(d) for d in search]}')
        return None
    numbered.sort(key=lambda x: x[1])
    print(f'Checkpoint: {numbered[-1][0]}')
    return numbered[-1][0]


def load_model(ckpt_path, device='cuda'):
    net = RRDBNet6x(num_in_ch=CFG.NUM_BANDS, num_out_ch=CFG.NUM_BANDS,
                    num_feat=CFG.NUM_FEAT, num_block=CFG.NUM_BLOCK)
    state = torch.load(ckpt_path, map_location='cpu')
    key = 'params_ema' if 'params_ema' in state else 'params' if 'params' in state else None
    net.load_state_dict(state[key] if key else state)
    print(f'Loaded {"EMA" if key == "params_ema" else "regular"} weights')
    return net.to(device).eval()


def compute_psnr(pred, gt, crop_border=0):
    if crop_border > 0:
        pred = pred[:, crop_border:-crop_border, crop_border:-crop_border]
        gt   = gt[:,   crop_border:-crop_border, crop_border:-crop_border]
    mse = np.mean((pred - gt) ** 2)
    return 100.0 if mse < 1e-10 else 10 * np.log10(1.0 / mse)


def compute_sam(pred, gt):
    dot = np.sum(pred * gt, axis=0)
    np_ = np.linalg.norm(pred, axis=0).clip(1e-8)
    ng  = np.linalg.norm(gt,   axis=0).clip(1e-8)
    return np.degrees(np.arccos(np.clip(dot / (np_ * ng), -1 + 1e-7, 1 - 1e-7)).mean())


def evaluate_full(split='test', save=True, ckpt_path=None):
    """Full evaluation: per-tile PSNR/SAM + per-band spectral fidelity."""
    ckpt = Path(ckpt_path) if ckpt_path else find_checkpoint(CFG.EXP_DIR, CFG.EXP_NAME)
    if ckpt is None:
        return None
    net = load_model(ckpt)

    gt_files = sorted((CFG.LOCAL_DATA / split / 'gt').glob('*.npy'))
    lq_files = sorted((CFG.LOCAL_DATA / split / 'lq').glob('*.npy'))
    assert len(gt_files) == len(lq_files) > 0, f'No data in {CFG.LOCAL_DATA}/{split}'

    nb = CFG.NUM_BANDS
    ps_sr, ps_bic, sm_sr, sm_bic = [], [], [], []
    bm_sr  = np.zeros(nb); bm_bic = np.zeros(nb)
    br_sr  = np.zeros(nb); br_bic = np.zeros(nb)
    bt     = np.zeros(nb); bc_cnt = np.zeros(nb)

    for gp, lp in tqdm(zip(gt_files, lq_files), total=len(gt_files), desc=f'Eval {split}'):
        gt = np.load(gp).astype(np.float32)
        lq = np.load(lp).astype(np.float32)
        with torch.no_grad():
            sr = net(torch.from_numpy(lq[None]).float().cuda()).squeeze(0).cpu().numpy()
        bic = F.interpolate(torch.from_numpy(lq[None]), scale_factor=CFG.SCALE,
                            mode='bicubic', align_corners=False).squeeze(0).numpy()

        cb = CFG.SCALE
        sc  = sr[:,  cb:-cb, cb:-cb]
        bc2 = bic[:, cb:-cb, cb:-cb]
        gc  = gt[:,  cb:-cb, cb:-cb]

        ps_sr.append(compute_psnr(sc, gc))
        ps_bic.append(compute_psnr(bc2, gc))
        sm_sr.append(compute_sam(sc, gc))
        sm_bic.append(compute_sam(bc2, gc))

        for b in range(nb):
            ds = (sc[b] - gc[b]).ravel()
            db = (bc2[b] - gc[b]).ravel()
            g  = gc[b].ravel()
            bm_sr[b]  += np.sum(ds ** 2)
            bm_bic[b] += np.sum(db ** 2)
            br_sr[b]  += np.sum(ds ** 2)
            br_bic[b] += np.sum(db ** 2)
            bt[b]     += np.sum((g - g.mean()) ** 2)
            bc_cnt[b] += len(g)

    R = dict(
        n=len(gt_files),
        psnr_sr=ps_sr, psnr_bic=ps_bic,
        sam_sr=sm_sr,  sam_bic=sm_bic,
        band_rmse_sr  = np.sqrt(bm_sr  / bc_cnt),
        band_rmse_bic = np.sqrt(bm_bic / bc_cnt),
        band_psnr_sr  = 10 * np.log10(1.0 / (bm_sr  / bc_cnt + 1e-10)),
        band_psnr_bic = 10 * np.log10(1.0 / (bm_bic / bc_cnt + 1e-10)),
        band_r2_sr    = 1 - br_sr  / np.maximum(bt, 1e-10),
        band_r2_bic   = 1 - br_bic / np.maximum(bt, 1e-10),
    )

    ps = np.mean(R['psnr_sr']);  pb = np.mean(R['psnr_bic'])
    ss = np.mean(R['sam_sr']);   sb = np.mean(R['sam_bic'])

    print(f'\n{"=" * 55}')
    print(f'  {split.upper()} — {R["n"]} tiles')
    print(f'{"=" * 55}')
    print(f'  {"Metric":<12}{"SR":>12}{"Bicubic":>12}{"Δ":>10}')
    print(f'  {"-" * 46}')
    print(f'  {"PSNR (dB)":<12}{ps:>12.4f}{pb:>12.4f}{ps - pb:>+10.4f}')
    print(f'  {"SAM (deg)":<12}{ss:>12.4f}{sb:>12.4f}{ss - sb:>+10.4f}')
    print(f'  {"Mean RMSE":<12}{np.mean(R["band_rmse_sr"]):>12.6f}{np.mean(R["band_rmse_bic"]):>12.6f}')
    print(f'  {"Mean R²":<12}{np.mean(R["band_r2_sr"]):>12.6f}{np.mean(R["band_r2_bic"]):>12.6f}')

    # ── Distribution plots ──
    fig1, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.hist(R['psnr_sr'],  bins=30, alpha=.7, color='#2196F3', label='SR')
    a1.hist(R['psnr_bic'], bins=30, alpha=.7, color='#FF9800', label='Bicubic')
    a1.axvline(ps, color='#1565C0', ls='--', lw=2, label=f'SR={ps:.2f}')
    a1.axvline(pb, color='#E65100', ls='--', lw=2, label=f'Bic={pb:.2f}')
    a1.set(xlabel='PSNR (dB)', ylabel='Count', title='Per-tile PSNR')
    a1.legend(fontsize=8); a1.grid(True, alpha=.3)
    a2.hist(R['sam_sr'],  bins=30, alpha=.7, color='#2196F3', label='SR')
    a2.hist(R['sam_bic'], bins=30, alpha=.7, color='#FF9800', label='Bicubic')
    a2.axvline(ss, color='#1565C0', ls='--', lw=2, label=f'SR={ss:.2f}°')
    a2.axvline(sb, color='#E65100', ls='--', lw=2, label=f'Bic={sb:.2f}°')
    a2.set(xlabel='SAM (degrees)', ylabel='Count', title='Per-tile SAM')
    a2.legend(fontsize=8); a2.grid(True, alpha=.3)
    plt.suptitle(f'{split} — {R["n"]} tiles', fontsize=13); plt.tight_layout()

    # ── Spectral plots ──
    fig2, (b1, b2, b3) = plt.subplots(1, 3, figsize=(18, 5))
    bands_x = np.arange(nb)
    for ax, k, yl, t in [(b1, 'band_psnr', 'PSNR (dB)', 'Per-band PSNR'),
                          (b2, 'band_rmse', 'RMSE', 'Per-band RMSE'),
                          (b3, 'band_r2', 'R²', 'Per-band R²')]:
        ax.plot(bands_x, R[f'{k}_bic'], 'o-', ms=4, alpha=.7, color='#FF9800', label='Bicubic')
        ax.plot(bands_x, R[f'{k}_sr'],  's-', ms=4, alpha=.7, color='#2196F3', label='SR')
        ax.set(xlabel='Band', ylabel=yl, title=t); ax.legend(); ax.grid(True, alpha=.3)
    b3.set_ylim(0, 1.05)
    plt.suptitle(f'Spectral Fidelity — {split}', fontsize=13); plt.tight_layout()

    # ── Save ──
    if save:
        out = CFG.EXP_DIR / 'figures'
        out.mkdir(parents=True, exist_ok=True)
        fig1.savefig(out / f'distributions_{split}.png', dpi=150, bbox_inches='tight')
        fig2.savefig(out / f'spectral_{split}.png', dpi=150, bbox_inches='tight')
        with open(out / f'metrics_{split}.csv', 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['band', 'psnr_sr', 'psnr_bic', 'rmse_sr', 'rmse_bic', 'r2_sr', 'r2_bic'])
            for b in range(nb):
                w.writerow([b, f'{R["band_psnr_sr"][b]:.4f}', f'{R["band_psnr_bic"][b]:.4f}',
                            f'{R["band_rmse_sr"][b]:.6f}', f'{R["band_rmse_bic"][b]:.6f}',
                            f'{R["band_r2_sr"][b]:.6f}', f'{R["band_r2_bic"][b]:.6f}'])
        print(f'\nExported → {out}/')
    plt.show()
    return R


results = evaluate_full(split='test')

In [ ]:
def plot_visual(split='val', n_tiles=4, ckpt_path=None):
    """Visual comparison grid: Bicubic | SR | GT + error maps."""
    ckpt = Path(ckpt_path) if ckpt_path else find_checkpoint(CFG.EXP_DIR, CFG.EXP_NAME)
    if ckpt is None:
        return
    net = load_model(ckpt)

    gt_files = sorted((CFG.LOCAL_DATA / split / 'gt').glob('*.npy'))
    lq_files = sorted((CFG.LOCAL_DATA / split / 'lq').glob('*.npy'))
    if not gt_files:
        print(f'No data in {split}'); return

    idx = np.linspace(0, len(gt_files) - 1, min(n_tiles, len(gt_files)), dtype=int)
    print(f'{len(gt_files)} tiles in "{split}", showing {len(idx)}')

    for i in idx:
        gt = np.load(gt_files[i]).astype(np.float32)
        lq = np.load(lq_files[i]).astype(np.float32)
        with torch.no_grad():
            sr = net(torch.from_numpy(lq[None]).float().cuda()).squeeze(0).cpu().numpy()

        fig = make_vis_figure(lq, sr, gt, CFG.SCALE, CFG.VIS_BANDS)
        fig.suptitle(Path(gt_files[i]).stem, fontsize=8, y=1.01)
        plt.show()
        plt.close(fig)


plot_visual(split='test', n_tiles=6)